# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [1]:
# Maaz
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Joe
import polars as pl 
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random
from utils import *
from data.simulate_walk_the_book import *
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cpu


In [2]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


# Preprocessing

In [3]:
# Run this cell to see specifics on preprocessing
help(preprocess)

Help on function preprocess in module utils:

preprocess(X, y, device) -> tuple
    Clean, validate, normalize, and tensorize feature and target datasets.
    
    This function applies a deterministic preprocessing pipeline to input
    feature (`X`) and target (`y`) Polars DataFrames. The pipeline enforces
    strict temporal consistency per instrument, handles missing values using
    market-aware rules, removes invalid instruments, converts data into
    fixed-length PyTorch tensors, and applies z-score normalization.
    
    Processing steps:
    1. Sort data by instrument ID and time.
    2. Backfill only *leading* NaNs in price columns per instrument.
    3. Forward-fill remaining price NaNs.
    4. Replace missing volume values with zeros.
    5. Remove instruments with:
       - duplicate (ID, time) pairs
       - missing timesteps (incomplete sequences)
    6. Convert DataFrames into tensors of shape:
       - X: (input_seq_len, num_ids, num_features)
       - y: (target_seq

## 1. The Encoder: DeepLOB (Heavy)
This is the standard DeepLOB CNN-Inception encoder.

In [12]:
class DeepLOBEncoder(nn.Module):
    def __init__(self, in_ch: int = 1):
        super().__init__()
        # Conv Blocks 1, 2, 3 (Keep these exactly as you have them)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(1,2), stride=(1,1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2),stride=(1,1)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        # Inception Modules (Keep these exactly as you have them)
        self.inp1 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(3,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(5,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )

    def forward(self, x):
        # x: [Batch, Time, Features]
        B, T, F = x.shape
        
        # 1. Standardize to 20 features (5 levels * 4)
        x = x[:, :, :20] 
        
        # 2. Reshape for CNN: [B*T, 1, 5, 4]
        x = x.reshape(B * T, 1, 5, 4) 
        
        # 3. Convolutional Stages
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        
        # 4. Inception Stage
        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1) # [B*T, 192, 5, 1]
        
        # 5. Global Average Pooling (The Fix)
        # This collapses the '5' (height) into 1, giving us [B*T, 192, 1, 1]
        x = torch.mean(x, dim=(2, 3)) 
        
        # 6. Final Shape for LSTM: [Batch, Time, 192]
        return x.view(B, T, 192)

## 2. The Decoder: Seq2Seq Attention
This replaces the simple Bi-LSTM head with a full autoregressive decoder.

In [13]:
class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int = 64):
        super().__init__()
        self.W_e = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_d = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_outputs: torch.Tensor, dec_state: torch.Tensor) -> torch.Tensor:
        # enc_outputs: [B, T, E], dec_state: [B, D]
        e = self.W_e(enc_outputs) + self.W_d(dec_state).unsqueeze(1)  # [B, T, A]
        scores = self.v(torch.tanh(e)).squeeze(-1)  # [B, T]
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights.unsqueeze(-1) * enc_outputs, dim=1)  # [B, E]
        return context

class SuperModel(nn.Module):
    def __init__(self, hidden: int = 128, horizon: int = 60, dropout: float=0.1, ask_bid_idx=(0, 2)):
        super().__init__()
        # 1. Encoder Part
        self.spatial = DeepLOBEncoder(in_ch=1) # The Heavy CNN [B, T, 192]
        self.encoder = nn.LSTM(
            input_size=192, # DeepLOB output dim
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # 2. Decoder Part
        self.attn = AdditiveAttention(enc_dim=2 * hidden, dec_dim=hidden)
        self.decoder = nn.LSTM(
            input_size=1 + 2 * hidden,  # previous y + context
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )
        self.out = nn.Linear(hidden, 1)
        self.init_h = nn.Linear(2 * hidden, hidden)
        self.init_c = nn.Linear(2 * hidden, hidden)
        self.horizon = horizon

        self.ask_idx, self.bid_idx = ask_bid_idx

    def forward(self, x: torch.Tensor, y_teacher: torch.Tensor = None) -> torch.Tensor:
        # x: [B, T, F]
        # y_teacher: [B, 60]
        
        # Encode
        h_spatial = self.spatial(x)  # [B, T, 192]
        enc_out, _ = self.encoder(h_spatial)  # [B, T, 2H]

        # Init decoder state from encoder mean
        enc_mean = enc_out.mean(dim=1)
        dec_h0 = torch.tanh(self.init_h(enc_mean))
        dec_c0 = torch.tanh(self.init_c(enc_mean))
        dec_h = dec_h0.unsqueeze(0)  # [1, B, H]
        dec_c = dec_c0.unsqueeze(0)

        outputs = []
        # Seed: use last encoder input midprice proxy (col 0 normalized)
        last_ask = x[:, -1, self.ask_idx]
        last_bid = x[:, -1, self.bid_idx]
        prev_y = ((last_ask + last_bid) / 2.0).unsqueeze(-1) # [B, 1]

        for t in range(self.horizon):
            context = self.attn(enc_out, dec_h[-1])  # [B, 2H]
            dec_in = torch.cat([prev_y, context], dim=1).unsqueeze(1)  # [B, 1, 1+2H]
            dec_out, (dec_h, dec_c) = self.decoder(dec_in, (dec_h, dec_c))
            y_hat = self.out(dec_out).squeeze(1)  # [B, 1]
            outputs.append(y_hat.squeeze(-1))
            if self.training and y_teacher is not None:
                # Teacher forcing
                prev_y = y_teacher[:, t].unsqueeze(-1)
            else:
                # Autoregressive
                prev_y = y_hat.detach()

        return torch.stack(outputs, dim=1)  # [B, 60]

def forecast_loss(pred, target, smooth_lambda=0.02):
    mse = (pred - target) ** 2
    loss = mse.mean()
    if smooth_lambda > 0:
        smooth = (pred[:, 1:] - pred[:, :-1]) ** 2
        loss = loss + smooth_lambda * smooth.mean()
    return loss


## 3. Dataset & Training Loop

In [14]:
from dataclasses import dataclass
import pandas as pd
import polars as pl
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

@dataclass
class TrainCfg:
    epochs: int = 15
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-5
    smooth_lambda: float = 0.02
    val_ratio: float = 0.2
    input_window: int = 300   # Look-back period
    target_window: int = 60   # Prediction horizon

class TensorTimeDataset(Dataset):
    def __init__(self, X, Y_full, Y_mid, input_window: int):
        self.X = X      
        self.Y = Y_full 
        self.Y_mid = Y_mid 
        self.input_window = input_window

    def __len__(self):
        return self.X.shape[1]

    def __getitem__(self, idx):
        # Dynamically slice based on config input_window
        x_sample = self.X[-self.input_window:, idx, :] 
        y_full_sample = self.Y[:, idx, :]
        target = self.Y_mid[idx, :]
        return x_sample, y_full_sample, target

def chrono_split(x_df, y_df, val_ratio=0.2):
    ids = np.sort(x_df["anonymized_id"].unique())
    split = int(len(ids) * (1 - val_ratio))
    tr_ids, va_ids = ids[:split], ids[split:]
    
    x_tr = x_df[x_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    x_va = x_df[x_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_tr = y_df[y_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    y_va = y_df[y_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    return x_tr, x_va, y_tr, y_va

def train_epoch(model, loader, opt, cfg: TrainCfg):
    model.train()
    total_loss, n = 0.0, 0
    for xb, _, target in loader:
        xb, target = xb.to(device), target.to(device)
        opt.zero_grad()
        
        # SuperModel handles teacher forcing internally if y_teacher is provided
        pred = model(xb, y_teacher=target)
        loss = forecast_loss(pred, target, cfg.smooth_lambda)
        
        loss.backward()
        opt.step()
        
        total_loss += float(loss.item()) * xb.size(0)
        n += xb.size(0)
    return total_loss / max(n, 1)

def eval_epoch(model, loader, cfg: TrainCfg):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for xb, _, target in loader:
            xb, target = xb.to(device), target.to(device)
            # No teacher forcing during evaluation
            pred = model(xb, y_teacher=None)
            loss = forecast_loss(pred, target, cfg.smooth_lambda)
            total_loss += float(loss.item()) * xb.size(0)
            n += xb.size(0)
    return total_loss / max(n, 1)

def train_val(cfg: TrainCfg = TrainCfg()):
    # 1. Load Data
    X_raw = pd.read_parquet(X_path).sort_values(["anonymized_id", "time_in_hour"])
    Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
    
    # 2. Split
    x_tr_pd, x_va_pd, y_tr_pd, y_va_pd = chrono_split(X_raw, Y_raw, val_ratio=cfg.val_ratio)

    # 3. Preprocess (to Polars -> to Tensors)
    X_tr_tens, Y_tr_tens, tr_means, tr_stds, _, _, f_map = preprocess(
        pl.from_pandas(x_tr_pd), pl.from_pandas(y_tr_pd), device
    )
    X_va_tens, Y_va_tens, _, _, _, y_va_map, _ = preprocess(
        pl.from_pandas(x_va_pd), pl.from_pandas(y_va_pd), device
    )
    
    # 4. Target Mid-Price extraction
    a_idx, b_idx = f_map["ask_price_1"], f_map["bid_price_1"]
    mid_tr = (Y_tr_tens[:, :, a_idx] + Y_tr_tens[:, :, b_idx]) / 2.0
    mid_va = (Y_va_tens[:, :, a_idx] + Y_va_tens[:, :, b_idx]) / 2.0

    mid_mean = float(mid_tr.mean())
    mid_std  = float(mid_tr.std()) if mid_tr.std() != 0 else 1e-6

    # 5. Datasets (passing cfg.input_window)
    train_ds = TensorTimeDataset(X_tr_tens, Y_tr_tens, mid_tr.T, input_window=cfg.input_window)
    val_ds   = TensorTimeDataset(X_va_tens, Y_va_tens, mid_va.T, input_window=cfg.input_window)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

    # 6. Model & Training
    # Inside train_val, after getting f_map:
    ask_idx = f_map["ask_price_1"]
    bid_idx = f_map["bid_price_1"]

    model = SuperModel(
        horizon=cfg.target_window, 
        ask_bid_idx=(ask_idx, bid_idx)
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for epoch in range(cfg.epochs):
        tr_l = train_epoch(model, train_loader, optimizer, cfg)
        va_l = eval_epoch(model, val_loader, cfg)
        print(f"Epoch {epoch+1:02d} | Train: {tr_l:.6f} | Val: {va_l:.6f}")

    scalers = {"feat_means": tr_means, "feat_stds": tr_stds, "mid_mean": mid_mean, "mid_std": mid_std}
    return model, scalers, val_loader, y_va_map


In [ ]:
# --- Execution ---
config = TrainCfg(
    epochs=20, 
    batch_size=32, 
    input_window=5*60, # -> input window of 5 minutes
    smooth_lambda=0.01
)

# Standard DeepLOB expects 10 Ask (Price/Vol) and 10 Bid (Price/Vol)
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS += [f"ask_price_{i}", f"ask_vol_{i}", f"bid_price_{i}", f"bid_vol_{i}"]

# Ensure FEATURE_COLS matches exactly what the Encoder expects
FEATURE_COLS = LOB_COLS

model, scalers, val_loader, va_id_map = train_val(config)

## 4. Evaluation
Calculate implementation error (bps) usage `alpha=1.0` (pure model).

In [ ]:
from data.simulate_walk_the_book import simulate_walk_the_book
try:
    from predictive_scheduler import schedule_from_forecasts
except ImportError:
    from predictive_scheduler import build_schedule_from_forecasts as schedule_from_forecasts

run_eval = True

if run_eval and X_path.exists() and Y_path.exists() and 'model' in globals():
    # 1. Load Data for Validation
    X_full = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y_full = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)

    # Right before preprocessing
    FEATURE_COLS = [c for c in X_raw.columns if c not in ["anonymized_id", "time_in_hour"]]
    num_features = len(FEATURE_COLS)
    print(f"Total features detected: {num_features}")
    
    # Identify the validation set IDs (aligning with the 0.8 split in train_val)
    all_ids = np.sort(X_full["anonymized_id"].unique())
    split_idx = int(len(all_ids) * (1 - config.val_ratio))
    va_ids = all_ids[split_idx:]
    
    x_va_raw = X_full[X_full["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_va_raw = Y_full[Y_full["anonymized_id"].isin(va_ids)].reset_index(drop=True)

    # 2. Create an ID-to-Index map from the tensor returned by preprocess
    # va_id_map was the 4th return value of train_val
    id_to_idx = {int(uid): i for i, uid in enumerate(va_id_map.cpu().numpy())}

    errs = []
    model.eval()
    
    # Filter va_ids to only those that survived the preprocess filtering
    valid_eval_ids = [hid for hid in va_ids if int(hid) in id_to_idx]

    print(f"Evaluating {len(valid_eval_ids)} hours...")

    for hid in valid_eval_ids:
        # Get tensor index for this ID to fetch correct normalization stats
        idx = id_to_idx[int(hid)]
        
        xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        
        if len(yh_raw) != config.target_window:
            continue
            
        # 3. Normalization (Per-ID stats for features, Global stats for midprice)
        h_feat_means = scalers['feat_means'][0, idx, :].cpu().numpy()
        h_feat_stds  = scalers['feat_stds'][0, idx, :].cpu().numpy()
        mid_mean     = scalers['mid_mean']
        mid_std      = scalers['mid_std']
        
        # Prepare Input Window
        xh = xh_raw.tail(config.input_window)
        x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
        
        # Padding if history is shorter than window
        if x_arr.shape[0] < config.input_window:
            pad = np.zeros((config.input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
            x_arr = np.vstack([pad, x_arr])
            
        # Z-score using local stats
        x_arr = (x_arr - h_feat_means) / h_feat_stds
        xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        
        # 4. Model Prediction
        with torch.no_grad():
            # No teacher forcing during eval
            pred = model(xb, y_teacher=None)
            
        # Un-normalize back to price space
        mid_pred = (pred.cpu().numpy()[0] * mid_std) + mid_mean

        # 5. Build Trading Positions
        def end_window_twap(vol, window=14):
            w = np.zeros(60, dtype=np.float32); w[-window:] = vol / window; return w
            
        sched_forecast = schedule_from_forecasts(mid_pred, volume_to_fill)
        sched_twap = end_window_twap(volume_to_fill, window=14)
        
        alpha = 1.0  # Full model confidence
        positions = alpha * sched_forecast + (1-alpha) * sched_twap
        
        # Re-scale to ensure we fill exactly the volume_to_fill
        if positions.sum() != 0:
            positions = positions * (volume_to_fill / positions.sum())
        else:
            positions = sched_twap

        # 6. Market Simulation (Walk the Book)
        ask_prices = yh_raw[[f"ask_price_{lvl}" for lvl in range(1,6)]].to_numpy(dtype=float)
        ask_vols   = yh_raw[[f"ask_vol_{lvl}" for lvl in range(1,6)]].to_numpy(dtype=float)
        bid_prices = yh_raw[[f"bid_price_{lvl}" for lvl in range(1,6)]].to_numpy(dtype=float)
        bid_vols   = yh_raw[[f"bid_vol_{lvl}" for lvl in range(1,6)]].to_numpy(dtype=float)
        
        total_vol, avg_price = simulate_walk_the_book(positions, ask_prices, ask_vols, bid_prices, bid_vols)
        
        # 7. Error Metrics (BPS)
        close_price = yh_raw["close"].dropna().iloc[-1] if len(yh_raw["close"].dropna())>0 else np.nan
        
        if total_vol > 0 and not np.isnan(avg_price) and not np.isnan(close_price):
            rel_err = abs(avg_price - close_price) / close_price
            # Fill penalty: If we didn't fill everything, we get penalized
            fill_penalty = min(100.0, volume_to_fill / total_vol)
            err_bps = rel_err * fill_penalty * 10000.0
            errs.append(err_bps)

    # Final Statistics
    if errs:
        print(f"\n--- Evaluation Results ---")
        print(f"Hours evaluated: {len(errs)}")
        print(f"Mean BPS Error:   {np.mean(errs):.4f}")
        print(f"Median BPS Error: {np.median(errs):.4f}")
        print(f"Std Dev BPS:      {np.std(errs):.4f}")
    else:
        print("No valid error metrics could be calculated.")
else:
    print("Evaluation skipped. Check if X_path/Y_path exist and model is trained.")

Validation implementation error (bps) over 2 hours: mean=0.7898, median=0.7898


In [ ]:
# R^2 Score Calculation
run_r2 = True

if run_r2 and X_path.exists() and Y_path.exists() and 'model' in globals():
    from sklearn.metrics import r2_score
    
    # 1. Load and Split
    X_full = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y_full = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)

    # Right before preprocessing
    FEATURE_COLS = [c for c in X_full.columns if c not in ["anonymized_id", "time_in_hour"]]
    num_features = len(FEATURE_COLS)
    print(f"Total features detected: {num_features}")
    
    all_ids = np.sort(X_full["anonymized_id"].unique())
    split_idx = int(len(all_ids) * (1 - config.val_ratio))
    va_ids = all_ids[split_idx:]
    
    x_va_raw = X_full[X_full["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_va_raw = Y_full[Y_full["anonymized_id"].isin(va_ids)].reset_index(drop=True)

    # 2. Setup mapping and containers
    id_to_idx = {int(uid): i for i, uid in enumerate(va_id_map.cpu().numpy())}
    valid_eval_ids = [hid for hid in va_ids if int(hid) in id_to_idx]
    
    all_preds = []
    all_targets = []

    model.eval()
    print(f"Calculating R^2 for {len(valid_eval_ids)} sequences...")

    for hid in valid_eval_ids:
        # Get tensor index for correct ID-level normalization stats
        idx = id_to_idx[int(hid)]
        
        xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        
        if len(yh_raw) != config.target_window:
            continue
        
        # 3. Get Ground Truth Midprice (Real Price Space)
        mid_true = ((yh_raw["ask_price_1"] + yh_raw["bid_price_1"]) / 2.0).to_numpy()

        # 4. Prepare Normalized Input
        h_feat_means = scalers['feat_means'][0, idx, :].cpu().numpy()
        h_feat_stds  = scalers['feat_stds'][0, idx, :].cpu().numpy()
        mid_mean     = scalers['mid_mean']
        mid_std      = scalers['mid_std']
        
        xh = xh_raw.tail(config.input_window)
        x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
        
        # Pad if history is shorter than the configured input window
        if x_arr.shape[0] < config.input_window:
            pad = np.zeros((config.input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
            x_arr = np.vstack([pad, x_arr])
        
        # Apply local Z-score normalization
        x_arr = (x_arr - h_feat_means) / h_feat_stds
        xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        
        # 5. Predict and Un-normalize
        with torch.no_grad():
            # Standard inference: no teacher forcing
            pred_norm = model(xb, y_teacher=None)
        
        # Transform prediction from Z-score back to real price space
        mid_pred = (pred_norm.cpu().numpy()[0] * mid_std) + mid_mean
        
        all_preds.append(mid_pred)
        all_targets.append(mid_true)
    
    # 6. Aggregate and Score
    if all_preds:
        # Flatten: treating all 60 seconds across all hours as a single series
        flat_preds = np.concatenate(all_preds)
        flat_targets = np.concatenate(all_targets)
        
        # Mask NaNs to prevent calculation errors
        mask = np.isfinite(flat_preds) & np.isfinite(flat_targets)
        valid_preds = flat_preds[mask]
        valid_targets = flat_targets[mask]
        
        if len(valid_preds) > 0:
            r2 = r2_score(valid_targets, valid_preds)
            print(f"\n--- Statistical Performance ---")
            print(f"Validation R^2 Score: {r2:.4f}")
            print(f"Total Points:         {len(valid_preds)}")
            
            # Additional useful metric: Mean Absolute Error in price terms
            mae = np.mean(np.abs(valid_preds - valid_targets))
            print(f"Mean Absolute Error:  {mae:.6f}")
        else:
            print("Error: No valid predictions/targets found after NaN masking.")
    else:
        print("No predictions were made. Check sequence lengths and IDs.")
else:
    print("Set run_r2=True and ensure model/paths are ready.")

Validation R^2 Score (all 60s steps): -3.0373
(Evaluated on 120 valid points, 7500 NaNs removed)
